# Heuristic ablation

Comparison of near-uniform and VND allocations with greedy and beam-search tree construction. All variants use exact $P/Q$ error evaluation.

In [ ]:
# Resolve the project whether Jupyter starts here or from the repository root.
from collections import Counter, defaultdict
from hashlib import sha256
import importlib
import json
from pathlib import Path
from statistics import fmean
import sys
from time import perf_counter
from zlib import crc32

import matplotlib.pyplot as plt
from IPython.display import Markdown, display

working = Path.cwd().resolve()
PROJECT = working / "problem-2-min-error-seq" if (working / "problem-2-min-error-seq").is_dir() else working
SOURCE = PROJECT / "src"
if str(SOURCE) not in sys.path:
    sys.path.insert(0, str(SOURCE))

analysis_module = importlib.reload(importlib.import_module("analysis"))
heuristic_module = importlib.reload(importlib.import_module("heuristic"))
from greedy import build_greedy_tree
from input_data import load_phase_input
from strategy import OutcomeModel, SearchTimeout

select_cases = analysis_module.select_cases
write_csv = analysis_module.write_csv

## Settings

The deterministic test sample contains 180 cases per dimension and covers every $(K,M)$ pair. Checkpointing supports resumed execution.

In [ ]:
RUN_MODE = "test"
BEAM_WIDTH = 3
MAX_ITERATIONS = 30
MOVE_SIZES = (1, 2, 3)
TIME_LIMIT_SECONDS = 30.0
RESUME = True
STUDY_VERSION = 1

METHODS = (
    "Near-uniform + Greedy",
    "VND allocation + Greedy",
    "Near-uniform + Beam",
    "VND allocation + Beam",
)
METHOD_KEYS = {
    "Near-uniform + Greedy": "near_uniform_greedy",
    "VND allocation + Greedy": "vnd_greedy",
    "Near-uniform + Beam": "near_uniform_beam",
    "VND allocation + Beam": "vnd_beam",
}
METHOD_STYLES = {
    "Near-uniform + Greedy": ("#9467bd", "D", ":"),
    "VND allocation + Greedy": ("#d62728", "s", "--"),
    "Near-uniform + Beam": ("#1f77b4", "o", "-"),
    "VND allocation + Beam": ("#2ca02c", "^", "-."),
}

OUTPUT = PROJECT / "results" / "heuristic_ablation" / RUN_MODE
CHECKPOINT = OUTPUT / "records.jsonl"

print("Run mode:", RUN_MODE)
print("Beam width:", BEAM_WIDTH)
print("Output:", OUTPUT)

## Ablation methods

Each case and budget uses one near-uniform allocation and one VND allocation. Both allocations are evaluated with greedy and beam-search trees.

In [ ]:
def vnd_allocation(model, *, random_seed):
    allocation = heuristic_module._initial_allocation(
        model.M, model.N, "near_uniform", random_seed
    )
    _, pair_errors = heuristic_module._compute_pair_errors(
        model.pairs, model.M, allocation, model.phases
    )
    score = heuristic_module._allocation_score(pair_errors)
    level = 0
    iterations = 0

    while level < len(MOVE_SIZES):
        model.check_time()
        if iterations >= MAX_ITERATIONS:
            break
        iterations += 1
        candidate = heuristic_module._best_allocation_neighbor(
            allocation,
            model.pairs,
            model.M,
            model.phases,
            score,
            MOVE_SIZES[level],
            0.0,
        )
        if candidate is None:
            level += 1
        else:
            allocation, score = candidate
            level = 0
    model.check_time()
    return allocation, iterations


def evaluate_greedy_tree(K, M, N, phases, allocation):
    model = OutcomeModel(K, M, N, phases, time_limit=TIME_LIMIT_SECONDS)
    tree = build_greedy_tree(model, allocation)
    return model.result(allocation, tree)


def evaluate_beam_tree(K, M, N, phases, allocation, beam_width):
    model = OutcomeModel(K, M, N, phases, time_limit=TIME_LIMIT_SECONDS)
    pair_locations, pair_errors = heuristic_module._compute_pair_errors(
        model.pairs, model.M, allocation, model.phases
    )
    pair_outcomes = {
        pair: model.probabilities(
            pair,
            pair_locations[pair],
            allocation[pair_locations[pair]],
        )
        for pair in model.pairs
    }
    tree = heuristic_module._build_tree_with_beam_search(
        K,
        pair_locations,
        pair_errors,
        pair_outcomes,
        beam_width,
        check_time=model.check_time,
    )
    return model.result(allocation, tree)


def timed_result(function):
    started = perf_counter()
    try:
        result = function()
        return "ok", result, perf_counter() - started
    except SearchTimeout:
        return "timeout", None, perf_counter() - started


def load_checkpoint(path):
    records = {}
    if not path.exists():
        return records
    for line in path.read_text().splitlines():
        try:
            record = json.loads(line)
            records[record["fingerprint"]] = record
        except (ValueError, KeyError):
            continue
    return records


def run_one(entry, settings_hash):
    dataset, path, metadata = entry
    K, M, phases = load_phase_input(path)
    records = []

    for budget_level, budget in enumerate(metadata["sensor_budgets"], 1):
        budget = int(budget)
        seed = crc32(f"{dataset}:{path.stem}:{budget}".encode())
        fingerprint = sha256(
            path.read_bytes()
            + settings_hash.encode()
            + json.dumps(metadata, sort_keys=True).encode()
            + str(budget).encode()
        ).hexdigest()
        records.append((fingerprint, budget_level, budget, K, M, phases, seed))
    return records


def calculate_record(dataset, path, metadata, item):
    fingerprint, budget_level, budget, K, M, phases, seed = item
    near_uniform = heuristic_module._initial_allocation(
        M, budget, "near_uniform", seed
    )

    allocation_model = OutcomeModel(
        K, M, budget, phases, time_limit=TIME_LIMIT_SECONDS
    )
    allocation_status, allocation_result, allocation_seconds = timed_result(
        lambda: vnd_allocation(allocation_model, random_seed=seed)
    )
    if allocation_status == "ok":
        vnd, vnd_iterations = allocation_result
    else:
        vnd, vnd_iterations = None, None

    variants = {
        "near_uniform_greedy": lambda: evaluate_greedy_tree(
            K, M, budget, phases, near_uniform
        ),
        "near_uniform_beam": lambda: evaluate_beam_tree(
            K, M, budget, phases, near_uniform, BEAM_WIDTH
        ),
    }
    if vnd is not None:
        variants.update({
            "vnd_greedy": lambda: evaluate_greedy_tree(
                K, M, budget, phases, vnd
            ),
            "vnd_beam": lambda: evaluate_beam_tree(
                K, M, budget, phases, vnd, BEAM_WIDTH
            ),
        })

    record = {
        "fingerprint": fingerprint,
        "dataset": dataset,
        "case": path.stem,
        "dimension": int(metadata["spatial_dimension"]),
        "field_family": metadata["field_model"],
        "replicate": int(metadata["replicate"]),
        "K": K,
        "M": M,
        "N": budget,
        "budget_level": budget_level,
        "budget_fraction": budget / int(metadata["minimum_sensors_for_zero_error"]),
        "near_uniform_allocation": list(near_uniform),
        "vnd_allocation": None if vnd is None else list(vnd),
        "vnd_iterations": vnd_iterations,
        "vnd_allocation_status": allocation_status,
        "vnd_allocation_seconds": allocation_seconds,
    }

    for method in METHODS:
        key = METHOD_KEYS[method]
        if key not in variants:
            status, result, seconds = "timeout", None, allocation_seconds
        else:
            status, result, seconds = timed_result(variants[key])
        record[f"{key}_status"] = status
        record[f"{key}_error"] = None if result is None else result.actual_error
        record[f"{key}_seconds"] = seconds
    return record


def run_ablation():
    entries = select_cases(PROJECT, "extended", RUN_MODE, exact_k_max=4)
    settings = {
        "study_version": STUDY_VERSION,
        "run_mode": RUN_MODE,
        "beam_width": BEAM_WIDTH,
        "max_iterations": MAX_ITERATIONS,
        "move_sizes": MOVE_SIZES,
        "time_limit_seconds": TIME_LIMIT_SECONDS,
    }
    code_hash = sha256(
        b"".join(p.read_bytes() for p in sorted(SOURCE.glob("*.py")))
    ).hexdigest()
    settings_hash = sha256(
        (json.dumps(settings, sort_keys=True) + code_hash).encode()
    ).hexdigest()
    cached = load_checkpoint(CHECKPOINT) if RESUME else {}
    OUTPUT.mkdir(parents=True, exist_ok=True)
    totals = Counter(dataset for dataset, _, _ in entries)
    completed = Counter()
    selected_records = []

    for dataset, path, metadata in entries:
        for item in run_one((dataset, path, metadata), settings_hash):
            fingerprint = item[0]
            if fingerprint in cached:
                record = cached[fingerprint]
            else:
                record = calculate_record(dataset, path, metadata, item)
                with CHECKPOINT.open("a") as stream:
                    stream.write(json.dumps(record, allow_nan=False) + "\n")
                    stream.flush()
                cached[fingerprint] = record
            selected_records.append(record)

        completed[dataset] += 1
        print(
            f"{dataset}: completed case {completed[dataset]}/{totals[dataset]} "
            f"({metadata['field_model']}, K={metadata['K']}, M={metadata['M']})",
            flush=True,
        )
    return selected_records


records = run_ablation()
print("Selected case-budget records:", len(records))

## Summary tables

Matched error and runtime results for the four ablation variants.

In [ ]:
def matched_records(records):
    return [
        record for record in records
        if all(record[f"{METHOD_KEYS[method]}_status"] == "ok" for method in METHODS)
    ]


def summarize(records, *, by_budget):
    groups = defaultdict(list)
    for record in matched_records(records):
        key = (
            int(record["dimension"]),
            int(record["budget_level"]) if by_budget else None,
        )
        groups[key].append(record)

    rows = []
    for (dimension, level), selected in sorted(groups.items()):
        row = {
            "dimension": dimension,
            "runs": len(selected),
        }
        if by_budget:
            row["budget_level"] = level
            row["mean_budget_fraction"] = fmean(
                float(record["budget_fraction"]) for record in selected
            )
        for method in METHODS:
            key = METHOD_KEYS[method]
            row[method] = fmean(float(record[f"{key}_error"]) for record in selected)
        row["allocation_gain_with_greedy_tree"] = (
            row["Near-uniform + Greedy"] - row["VND allocation + Greedy"]
        )
        row["beam_gain_with_near_uniform_allocation"] = (
            row["Near-uniform + Greedy"] - row["Near-uniform + Beam"]
        )
        row["beam_gain_with_vnd_allocation"] = (
            row["VND allocation + Greedy"] - row["VND allocation + Beam"]
        )
        row["full_gain"] = (
            row["Near-uniform + Greedy"] - row["VND allocation + Beam"]
        )
        rows.append(row)
    return rows


def markdown_table(rows):
    columns = list(rows[0])
    lines = [
        "| " + " | ".join(column.replace("_", " ").title() for column in columns) + " |",
        "|" + "|".join("---:" for _ in columns) + "|",
    ]
    for row in rows:
        values = []
        for column in columns:
            value = row[column]
            if isinstance(value, float):
                values.append(f"{value:.5f}")
            else:
                values.append(str(value))
        lines.append("| " + " | ".join(values) + " |")
    return "\n".join(lines)


matched = matched_records(records)
overall_rows = summarize(records, by_budget=False)
budget_rows = summarize(records, by_budget=True)

case_rows = [
    {key: value for key, value in record.items() if key != "fingerprint"}
    for record in records
]
timeout_rows = [
    {
        "dataset": record["dataset"],
        "case": record["case"],
        "dimension": record["dimension"],
        "K": record["K"],
        "M": record["M"],
        "N": record["N"],
        "method": method,
        "seconds": record[f"{METHOD_KEYS[method]}_seconds"],
    }
    for record in records
    for method in METHODS
    if record[f"{METHOD_KEYS[method]}_status"] != "ok"
]

write_csv(OUTPUT / "tables" / "case_results.csv", case_rows)
write_csv(OUTPUT / "tables" / "overall_summary.csv", overall_rows)
write_csv(OUTPUT / "tables" / "budget_summary.csv", budget_rows)
write_csv(
    OUTPUT / "tables" / "timeouts.csv",
    timeout_rows,
    fieldnames=("dataset", "case", "dimension", "K", "M", "N", "method", "seconds"),
)

display(Markdown("### Overall ablation summary"))
display(Markdown(markdown_table(overall_rows)))
display(Markdown("### Ablation summary by budget"))
display(Markdown(markdown_table(budget_rows)))
print("Matched records:", len(matched), "of", len(records))
print("Timeouts:", len(timeout_rows))

## Error curves

In [ ]:
def plot_ablation(rows):
    plt.rcParams.update({
        "font.family": "serif",
        "font.size": 11,
        "mathtext.fontset": "stix",
        "pdf.fonttype": 42,
        "axes.spines.top": False,
        "axes.spines.right": False,
    })
    figure, axes = plt.subplots(1, 2, figsize=(11.5, 4.2), sharey=True, layout="constrained")
    for dimension, axis in zip((1, 2), axes):
        selected = [row for row in rows if int(row["dimension"]) == dimension]
        for method in METHODS:
            color, marker, line = METHOD_STYLES[method]
            axis.plot(
                [row["mean_budget_fraction"] for row in selected],
                [row[method] for row in selected],
                color=color,
                marker=marker,
                linestyle=line,
                linewidth=1.8,
                label=method,
            )
        axis.set_title(
            "(a) One-dimensional dataset" if dimension == 1
            else "(b) Two-dimensional dataset",
            fontsize=12,
        )
        axis.set_xlabel(r"Mean budget fraction, $N/N_{\min}^{\mathrm{seq}}$")
        axis.grid(alpha=0.22)
    axes[0].set_ylabel("Mean error probability")
    axes[0].set_ylim(bottom=0)
    handles, labels = axes[0].get_legend_handles_labels()
    figure.legend(handles, labels, loc="outside upper center", ncol=2, frameon=False)
    return figure


figure = plot_ablation(budget_rows)
figure_directory = OUTPUT / "figures"
figure_directory.mkdir(parents=True, exist_ok=True)
for extension in ("pdf", "png"):
    figure.savefig(
        figure_directory / f"heuristic_ablation.{extension}",
        dpi=300,
        bbox_inches="tight",
    )
display(figure)
plt.close(figure)
print("Saved ablation outputs under:", OUTPUT)

## Saved outputs

Resumable records, tables, and figures are stored in `results/heuristic_ablation/test`.